In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("C://Users//dell//Downloads//my_file.csv")
 
print("--- BEFORE ---")
print(df.head())
print(df.dtypes)
print()

--- BEFORE ---
   Rank  Peak All Time Peak  Actual gross Adjusted gross (in 2022 dollars)  \
0     1     1             2  $780,000,000                     $780,000,000   
1     2     1          7[2]  $579,800,000                     $579,800,000   
2     3  1[4]          2[5]  $411,000,000                     $560,622,615   
3     4  2[7]         10[7]  $397,300,000                     $454,751,555   
4     5  2[4]           NaN  $345,675,146                     $402,844,849   

         Artist                   Tour title    Year(s)  Shows Average gross  \
0  Taylor Swift              The Eras Tour †  2023–2024     56   $13,928,571   
1       Beyoncé       Renaissance World Tour       2023     56   $10,353,571   
2       Madonna  Sticky & Sweet Tour ‡[4][a]  2008–2009     85    $4,835,294   
3          Pink  Beautiful Trauma World Tour  2018–2019    156    $2,546,795   
4  Taylor Swift      Reputation Stadium Tour       2018     53    $6,522,173   

  Ref.  
0  [1]  
1  [3]  
2  [6]  

In [3]:
# 1. Clean column names (fix hidden non-breaking spaces, standardize casing)
df.columns = (
    df.columns.astype(str)
    .str.replace("\xa0", " ", regex=False)   # non-breaking space -> normal space
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)  # drop punctuation like ( ) .
    .str.replace(r"\s+", "_", regex=True)     # spaces -> underscores
)
print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['rank', 'peak', 'all_time_peak', 'actual_gross', 'adjusted_gross_in_2022_dollars', 'artist', 'tour_title', 'years', 'shows', 'average_gross', 'ref']


In [4]:
# 2. Strip footnote references like [2], [4][a], [b] out of every text field
def strip_brackets(x):
    if pd.isna(x):
        return x
    return re.sub(r"\[.*?\]", "", str(x)).strip()
 
for col in ["peak", "all_time_peak", "actual_gross", "adjusted_gross_in_2022_dollars",
            "average_gross", "tour_title"]:
    df[col] = df[col].apply(strip_brackets)

In [5]:
# 3. Remove leftover footnote symbols (†, ‡, *) from tour titles
df["tour_title"] = df["tour_title"].str.replace(r"[†‡*]", "", regex=True).str.strip()

In [6]:
# 4. Convert Peak / All Time Peak to nullable integers
#  (blank = tour never reached that chart position, so left as missing, not 0)
df["peak"] = pd.to_numeric(df["peak"], errors="coerce").astype("Int64")
df["all_time_peak"] = pd.to_numeric(df["all_time_peak"], errors="coerce").astype("Int64")
 

In [7]:
# 5. Convert currency text ("$780,000,000") to numeric
def clean_currency(x):
    if pd.isna(x):
        return x
    return re.sub(r"[$,]", "", str(x))
 
df["actual_gross_usd"] = pd.to_numeric(df["actual_gross"].apply(clean_currency), errors="coerce")
df["adjusted_gross_2022_usd"] = pd.to_numeric(df["adjusted_gross_in_2022_dollars"].apply(clean_currency), errors="coerce")
df["average_gross_usd"] = pd.to_numeric(df["average_gross"].apply(clean_currency), errors="coerce")
 
df = df.drop(columns=["actual_gross", "adjusted_gross_in_2022_dollars", "average_gross"])
 

In [8]:
# 6. Split "Year(s)" (e.g. "2023" or "2023-2024", using an en-dash) into two int columns
def split_years(x):
    x = str(x).replace("\u2013", "-").replace("\u2014", "-")  # en-dash / em-dash -> hyphen
    parts = x.split("-")
    start = int(parts[0])
    end = int(parts[1]) if len(parts) > 1 else start
    return pd.Series([start, end])
 
df[["year_start", "year_end"]] = df["years"].apply(split_years)
df = df.drop(columns=["years"])

In [9]:
# 7. Fix the rank sequence
#    Original data has rank "7" appearing twice and rank "8" missing entirely —
#    a scrape/entry error. Re-derive rank from actual gross, descending.
df = df.sort_values("actual_gross_usd", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
 

In [10]:
# 8. Tidy the citation column
df = df.rename(columns={"ref": "citation_ref"})

In [11]:
# 9. Reorder columns for readability
# ---------------------------------------------------------------
df = df[["rank", "artist", "tour_title", "year_start", "year_end", "shows",
         "actual_gross_usd", "adjusted_gross_2022_usd", "average_gross_usd",
         "peak", "all_time_peak", "citation_ref"]]
 
print("\n--- AFTER ---")
print(df.head())
print(df.dtypes)
print("\nMissing values:\n", df.isna().sum())


--- AFTER ---
   rank        artist                   tour_title  year_start  year_end  \
0     1  Taylor Swift                The Eras Tour        2023      2024   
1     2       Beyoncé       Renaissance World Tour        2023      2023   
2     3       Madonna          Sticky & Sweet Tour        2008      2009   
3     4          Pink  Beautiful Trauma World Tour        2018      2019   
4     5  Taylor Swift      Reputation Stadium Tour        2018      2018   

   shows  actual_gross_usd  adjusted_gross_2022_usd  average_gross_usd  peak  \
0     56         780000000                780000000           13928571     1   
1     56         579800000                579800000           10353571     1   
2     85         411000000                560622615            4835294     1   
3    156         397300000                454751555            2546795     2   
4     53         345675146                402844849            6522173     2   

   all_time_peak citation_ref  
0              

In [12]:
df

,rank,artist,tour_title,year_start,year_end,shows,actual_gross_usd,adjusted_gross_2022_usd,average_gross_usd,peak,all_time_peak,citation_ref
0,1,Taylor Swift,The Eras Tour,2023,2024,56,780000000,780000000,13928571,1,2,[1]
1,2,Beyoncé,Renaissance World Tour,2023,2023,56,579800000,579800000,10353571,1,7,[3]
2,3,Madonna,Sticky & Sweet Tour,2008,2009,85,411000000,560622615,4835294,1,2,[6]
3,4,Pink,Beautiful Trauma World Tour,2018,2019,156,397300000,454751555,2546795,2,10,[7]
4,5,Taylor Swift,Reputation Stadium Tour,2018,2018,53,345675146,402844849,6522173,2,<NA>,[8]
5,6,Madonna,The MDNA Tour,2012,2012,88,305158363,388978496,3467709,2,10,[9]
6,7,Celine Dion,Taking Chances World Tour,2008,2009,131,280000000,381932682,2137405,2,<NA>,[11]
7,8,Pink,Summer Carnival,2023,2024,41,257600000,257600000,6282927,<NA>,<NA>,[12]
8,9,Beyoncé,The Formation World Tour,2016,2016,49,256084556,312258401,5226215,<NA>,<NA>,[13]
9,10,Taylor Swift,The 1989 World Tour,2015,2015,85,250400000,309141878,2945882,<NA>,<NA>,[14]
